# Multi-Layer Perceptron (Neural Network)

An MLP is a **feedforward neural network** trained with backpropagation and mini-batch gradient descent. It can learn non-linear decision boundaries through stacked hidden layers.

| Layer | Activation |
|-------|-----------|
| Hidden | Sigmoid |
| Output | Softmax |

**Loss:** Cross-entropy  
**Dataset:** MNIST digits — classify handwritten digits 0–9.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')
from multi_layer_perceptron import multi_layer_perceptron
np.random.seed(42)
print("Imports complete")

## Load the MNIST Dataset

We use sklearn's version (8x8 pixels, 64 features) for fast training.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features (8x8 pixels)")
print(f"Classes: {np.unique(y).tolist()} ({len(np.unique(y))} digit classes)")
print(f"Pixel range: [{X.min()}, {X.max()}]")

# Show sample digits
fig, axes = plt.subplots(2, 8, figsize=(14,4))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(str(label), fontsize=9)
    ax.axis('off')
plt.suptitle("Sample MNIST Digits", y=1.02)
plt.tight_layout()
plt.show()

## Preprocess & Split

In [ ]:
# Normalise to [0, 1]
X_norm = X / 16.0

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Class distribution (train): {dict(zip(*np.unique(y_train, return_counts=True)))}")

## Architecture Overview

We build a network: `64 → 128 → 64 → 10`
- Input: 64 features (8×8 pixels)
- Hidden 1: 128 neurons (sigmoid)
- Hidden 2: 64 neurons (sigmoid)
- Output: 10 neurons (softmax) — one per digit

In [ ]:
model = multi_layer_perceptron(
    layer_sizes=[64, 128, 64, 10],
    learning_rate=0.05,
    epochs=30,
    batch_size=32,
    random_state=42
)
model.fit(X_train, y_train)

## Training Loss Curve

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(range(1, len(model.loss_history)+1), model.loss_history,
         color='steelblue', linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("MLP Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Initial loss: {model.loss_history[0]:.4f} | Final loss: {model.loss_history[-1]:.4f}")

## Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

train_acc = model.score(X_train, y_train)
test_acc  = model.score(X_test,  y_test)
y_pred    = model.predict(X_test)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test  Accuracy: {test_acc:.4f}")
print()
print(classification_report(y_test, y_pred))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9,7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted Digit"); ax.set_ylabel("Actual Digit")
ax.set_title("Confusion Matrix — MLP on Digits")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=9,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Visualise Predictions on Test Samples

In [ ]:
# Show 16 test samples with true/predicted labels
probas = model.predict_proba(X_test)
fig, axes = plt.subplots(2, 8, figsize=(14,5))
indices = np.random.choice(len(X_test), 16, replace=False)
for ax, idx in zip(axes.ravel(), indices):
    img = X_test[idx].reshape(8, 8)
    pred = y_pred[idx]
    true = y_test[idx]
    ax.imshow(img, cmap='gray_r')
    color = 'green' if pred == true else 'red'
    ax.set_title(f'P:{pred} T:{true}', fontsize=9, color=color)
    ax.axis('off')
plt.suptitle("Test Predictions (Green=Correct, Red=Wrong)", y=1.02)
plt.tight_layout()
plt.show()

## Effect of Architecture — Depth & Width

In [ ]:
configs = {
    'Shallow [64,32,10]':      [64, 32, 10],
    'Default [64,128,64,10]':  [64, 128, 64, 10],
    'Narrow  [64,16,16,10]':   [64, 16, 16, 10],
    'Wide    [64,256,10]':     [64, 256, 10],
}
results = {}
for name, layers in configs.items():
    m = multi_layer_perceptron(layer_sizes=layers, learning_rate=0.05,
                               epochs=20, batch_size=32, random_state=42)
    m.fit(X_train, y_train)
    results[name] = m.score(X_test, y_test)
    print(f"{name}: {results[name]:.4f}")

plt.figure(figsize=(9,5))
bars = plt.bar(list(results.keys()), list(results.values()),
               color=['#aaa','steelblue','#e9c46a','#2a9d8f'], edgecolor='white')
for bar, acc in zip(bars, results.values()):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=9)
plt.ylim(0.8, 1.0)
plt.ylabel("Test Accuracy")
plt.title("MLP Architecture Comparison")
plt.xticks(rotation=12)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Key Takeaways

- MLPs learn hierarchical representations through stacked layers.
- **Sigmoid** (hidden) + **Softmax** (output) + **cross-entropy** (loss) is a standard classification setup.
- Deeper/wider networks can fit more complex patterns but require more data and tuning.
- Batch gradient descent with shuffling helps escape local minima.
